# LangChain: Code Execution (اجرای کد توسط Agent)

## Outline
1. چرا code execution؟ و ریسک‌هاش
2. `PythonREPLTool`: ساده‌ترین راه
3. `PythonAstREPLTool`: کار با داده (DataFrame)
4. ساخت یک tool اجرای کد با محدودیت زمان (subprocess)
5. Code Interpreter خود OpenAI (اجرا در sandbox ابری)
6. مقایسه‌ی روش‌ها و جمع‌بندی

## ۱. چرا Code Execution؟

مدل‌های زبانی تو **محاسبه‌ی دقیق**، **تحلیل داده**، **رسم نمودار** و کارهایی که نیاز به الگوریتم دارن ضعیفن یا اشتباه می‌کنن. اگه به agent اجازه‌ی اجرای کد بدیم، به‌جای «حدس زدن جواب» **کد می‌نویسه، اجرا می‌کنه و خروجی واقعی رو می‌بینه**؛ یعنی همون حلقه‌ی ReAct، فقط tool ما یک مفسر پایتونه.

مثال: «فاکتوریل ۵۰ چنده؟» یا «میانگین ستون X این جدول؟» رو بهتره کد جواب بده تا مدل.

> ⚠️ **هشدار امنیتی:** کدی که LLM می‌نویسه **غیرقابل اعتماده** (مثلاً با prompt injection از یک صفحه‌ی وب یا فایل ممکنه کد مخرب تولید بشه). اجرای مستقیم روی سیستم خودتون به فایل‌ها، کلیدها و شبکه‌تون دسترسی داره. برای آموزش و سیستم شخصی اوکیه؛ برای محصول واقعی حتماً از **sandbox** (کانتینر، E2B، Riza، یا Code Interpreter ابری) استفاده کنید.

## ۲. نصب و آماده‌سازی

```
pip install -U langchain langchain-openai langchain-experimental pandas
```

ابزارهای `PythonREPLTool` و `PythonAstREPLTool` تو پکیج `langchain-experimental` هستن.

In [48]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0.2)

def show_steps(agent, question):
    """مراحل agent رو چاپ می‌کنه: کد ارسالی به tool و خروجی اون"""
    for chunk in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        latest = chunk["messages"][-1]
        if isinstance(latest, HumanMessage):
            print(f"[User]: {latest.content}\n")
        elif isinstance(latest, AIMessage):
            if latest.tool_calls:
                for tc in latest.tool_calls:
                    code = tc["args"].get("query") or tc["args"].get("code") or tc["args"]
                    print(f"[Tool Call]: {tc['name']}\n{code}\n")
            elif latest.content:
                print(f"[AI]: {latest.content}")
        elif isinstance(latest, ToolMessage):
            print(f"[Tool Result]: {str(latest.content)[:300]}\n")


In [50]:
r = llm.invoke("تو کلمه‌ی 'strawberry' حرف r چند بار تکرار شده؟  بشمار.")
print(r.content)

در کلمه‌ی 'strawberry' حرف 'r' دو بار تکرار شده است.


## ۳. `PythonREPLTool`: ساده‌ترین راه

یک مفسر پایتون که هر کدی بهش بدیم اجرا می‌کنه. خروجی رو فقط اگه با `print` چاپ بشه برمی‌گردونه؛ پس به agent بگید حتماً نتیجه رو `print` کنه.

اول خود tool رو مستقیم تست می‌کنیم:

In [52]:
from langchain_experimental.tools import PythonREPLTool
llm = init_chat_model("gpt-5.2", model_provider="openai", temperature=0.2)

python_repl = PythonREPLTool()
print(python_repl.name, "|", python_repl.description)
print()
print(python_repl.invoke("import math\nprint(math.factorial(20))"))


Python_REPL | A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.

2432902008176640000



حالا به agent می‌دیم. مدل خودش کد می‌نویسه و اجرا می‌کنه:

In [54]:
agent_repl = create_agent(
    model=llm,
    tools=[python_repl],
    system_prompt=(
        "You are a helpful assistant that solves problems by writing and running Python code. "
        "Always use the Python tool for calculations. Always print() the final result. "
        "Answer in Persian."
    ),
)

show_steps(agent_repl, "فاکتوریل ۵۰ چنده و مجموع ارقامش چقدره؟")


[User]: فاکتوریل ۵۰ چنده و مجموع ارقامش چقدره؟

[Tool Call]: Python_REPL
import math
n=math.factorial(50)
print(n)
print(sum(map(int,str(n))))

[Tool Result]: 30414093201713378043612608166064768844377641568960512000000000000
216


[AI]: ```python
print(30414093201713378043612608166064768844377641568960512000000000000)
print(216)
```


مثال دوم: مسئله‌ای که مدل به‌تنهایی معمولاً اشتباه می‌کنه (شمردن حروف):

In [56]:
show_steps(agent_repl, "تو کلمه‌ی 'strawberry' حرف r چند بار تکرار شده؟ با کد بشمار.")


[User]: تو کلمه‌ی 'strawberry' حرف r چند بار تکرار شده؟ با کد بشمار.

[Tool Call]: Python_REPL
word = 'strawberry'
count_r = word.count('r')
print(count_r)

[Tool Result]: 3


[AI]: ```python
word = "strawberry"
count_r = word.count("r")
print(count_r)
```

خروجی:
```
3
```


## ۴. `PythonAstREPLTool`: کار با داده (DataFrame)

این نسخه شبیه قبلیه ولی دو تفاوت مهم داره:
- آخرین عبارت کد رو خودش `eval` می‌کنه و مقدارش رو برمی‌گردونه (لازم نیست `print` کنید)
- با `locals` می‌تونیم **متغیرهای آماده** (مثل یک DataFrame) رو بهش بدیم تا agent روی داده‌ی ما کار کنه

مثال: یک جدول فروش و سؤال تحلیلی روش.

In [58]:
import pandas as pd
from langchain_experimental.tools import PythonAstREPLTool

df = pd.DataFrame({
    "city":    ["Tehran", "Tehran", "Berlin", "Berlin", "Tokyo", "Tokyo"],
    "product": ["A", "B", "A", "B", "A", "B"],
    "sales":   [120, 80, 200, 150, 90, 300],
})

ast_repl = PythonAstREPLTool(locals={"df": df})

agent_df = create_agent(
    model=llm,
    tools=[ast_repl],
    system_prompt=(
        "You are a data analyst. A pandas DataFrame named `df` is already loaded in the Python tool "
        f"with columns {list(df.columns)}. Use the tool to answer questions about it. "
        "Do not invent data. Answer in Persian."
    ),
)

show_steps(agent_df, "مجموع فروش هر شهر چقدره و کدوم شهر بیشترین فروش رو داره؟")


[User]: مجموع فروش هر شهر چقدره و کدوم شهر بیشترین فروش رو داره؟

[Tool Call]: python_repl_ast
import pandas as pd
# compute total sales per city
city_sales = df.groupby('city', as_index=False)['sales'].sum().sort_values('sales', ascending=False)
city_sales, city_sales.iloc[0].to_dict()

[Tool Result]: (     city  sales
2   Tokyo    390
0  Berlin    350
1  Tehran    200, {'city': 'Tokyo', 'sales': 390})

[AI]: مجموع فروش هر شهر (جمع ستون `sales`):

- **Tokyo**: 390  
- **Berlin**: 350  
- **Tehran**: 200  

**بیشترین فروش** مربوط به **Tokyo** با مقدار **390** است.


> نکته: اسم DataFrame و ستون‌ها رو تو system prompt بگید تا agent حدس نزنه. برای دیتاست‌های بزرگ فقط `df.head()` و `df.dtypes` رو تو prompt بذارید، نه کل داده.

## ۵. ساخت یک tool اجرای کد با محدودیت زمان

`PythonREPLTool` اگه مدل کد بی‌نهایت (`while True`) بنویسه، کل نوت‌بوک رو قفل می‌کنه. یک tool ساده با `@tool` می‌سازیم که کد رو **تو یک پروسه‌ی جدا** (`subprocess`) و با **timeout** اجرا می‌کنه:

- کد تو پروسه‌ی جدا اجرا می‌شه، پس بی‌نهایت loop فقط اون پروسه رو قفل می‌کنه و بعد از timeout kill می‌شه
- تو یک پوشه‌ی موقت اجرا می‌شه
- خروجی محدود می‌شه تا context مدل پر نشه

> ⚠️ این **sandbox واقعی نیست**: کد هنوز به فایل‌ها و شبکه‌ی سیستم شما دسترسی داره. فقط جلوی قفل شدن رو می‌گیره.

In [60]:
import subprocess, sys, tempfile
from langchain.tools import tool

@tool
def run_python(code: str) -> str:
    """Execute Python code in a separate process and return its stdout/stderr.
    The code must print() anything it wants to return. Timeout: 10 seconds."""
    with tempfile.TemporaryDirectory() as tmp:
        try:
            proc = subprocess.run(
                [sys.executable, "-c", code],
                capture_output=True, text=True, timeout=10, cwd=tmp,
            )
        except subprocess.TimeoutExpired:
            return "Error: execution timed out after 10 seconds."
    out = (proc.stdout + proc.stderr).strip()
    return out[:2000] or "(no output — did you forget to print()?)"

# تست مستقیم
print(run_python.invoke({"code": "print(sum(range(1, 101)))"}))
print(run_python.invoke({"code": "while True: pass"}))


5050
Error: execution timed out after 10 seconds.


In [62]:
agent_safe = create_agent(
    model=llm,
    tools=[run_python],
    system_prompt=(
        "You solve problems by writing Python code and running it with the run_python tool. "
        "Always print() the result. If the code fails, read the error, fix the code and try again. "
        "Answer in Persian."
    ),
)

show_steps(agent_safe, "اعداد اول بین ۱ تا ۵۰ رو پیدا کن و مجموعشون رو بگو.")


[User]: اعداد اول بین ۱ تا ۵۰ رو پیدا کن و مجموعشون رو بگو.

[Tool Call]: run_python
def primes_upto(n):
    primes=[]
    for x in range(2,n+1):
        is_p=True
        for d in range(2,int(x**0.5)+1):
            if x%d==0:
                is_p=False
                break
        if is_p:
            primes.append(x)
    return primes
p=primes_upto(50)
print(p)
print(sum(p))


[Tool Result]: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
328

[AI]: اعداد اول بین ۱ تا ۵۰ عبارت‌اند از:
۲، ۳، ۵، ۷، ۱۱، ۱۳، ۱۷، ۱۹، ۲۳، ۲۹، ۳۱، ۳۷، ۴۱، ۴۳، ۴۷

مجموع آن‌ها:
۳۲۸


### خودترمیمی: agent با دیدن خطا کدش رو اصلاح می‌کنه

چون خطای پایتون (`stderr`) به شکل متن به مدل برمی‌گرده، agent می‌تونه اشتباهش رو ببینه و دوباره امتحان کنه. این یکی از مزیت‌های اصلی code execution نسبت به جواب مستقیم مدله.

In [64]:
show_steps(
    agent_safe,
    "با کتابخونه‌ی numpy یک آرایه‌ی ۱ تا ۱۰ بساز و میانگین و انحراف معیارش رو حساب کن. "
    "اگه خطا گرفتی خودت رفعش کن.",
)


[User]: با کتابخونه‌ی numpy یک آرایه‌ی ۱ تا ۱۰ بساز و میانگین و انحراف معیارش رو حساب کن. اگه خطا گرفتی خودت رفعش کن.

[Tool Call]: run_python
import numpy as np
arr = np.arange(1, 11)
mean = arr.mean()
std = arr.std()
print('آرایه:', arr)
print('میانگین:', mean)
print('انحراف معیار:', std)


[Tool Result]: Traceback (most recent call last):
  File "<string>", line 5, in <module>
  File "C:\Users\Alireza\.conda\envs\python3.12\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[Tool Call]: run_python
import numpy as np
arr = np.arange(1, 11)
mean = arr.mean()
std = arr.std()  # population std (ddof=0)
print('array=', arr)
print('mean=', mean)
print('std=', std)


[Tool Result]: array= [ 1  2  3  4  5  6  7  8  9 10]
mean= 5.5
std= 2.8722813232690143

[AI]: با `numpy` آرایه‌ی ۱ تا ۱۰ رو ساختم و میانگین و انحراف معیارش رو حساب کردم (انحراف معیار پیش‌فرض `numpy`

## ۶. Code Interpreter خود OpenAI (sandbox ابری)

OpenAI یک tool داخلی به اسم **code_interpreter** داره: کد تو **sandbox روی سرور OpenAI** اجرا می‌شه، نه روی سیستم شما. پس از نظر امنیتی خیلی بهتره و می‌تونه فایل و نمودار هم تولید کنه. LangChain اون رو با ارسال یک dict به `bind_tools` (و Responses API) پشتیبانی می‌کنه.

نکته‌ها:
- هزینه‌ی جدا داره (به‌ازای هر session کانتینر) و تو قیمت‌گذاری OpenAI ببینید
- مدل باید Responses API رو پشتیبانی کنه (مثل `gpt-4.1`، `gpt-4o`، `gpt-5` و...)
- خروجی یک لیست از content block هاست، نه فقط متن

> این سلول بر اساس مستندات LangChain و OpenAI نوشته شده؛ اگه با نسخه‌ی نصب‌شده‌ی شما اسم چیزی فرق داشت، مستندات `ChatOpenAI` رو ببینید.

In [66]:
from langchain_openai import ChatOpenAI

llm_ci = ChatOpenAI(model="gpt-4.1", use_responses_api=True)

llm_with_ci = llm_ci.bind_tools([
    {"type": "code_interpreter", "container": {"type": "auto"}}
])

response = llm_with_ci.invoke(
    "Compute the first 15 Fibonacci numbers using code and tell me their sum."
)

# خروجی شامل بلاک‌های مختلفه: کدی که اجرا شده و متن نهایی
for block in response.content:
    if isinstance(block, dict):
        kind = block.get("type")
        if kind == "code_interpreter_call":
            print("[Code Executed]:\n", block.get("code"), "\n")
        elif kind == "text":
            print("[AI]:", block.get("text"))
    else:
        print(block)


[Code Executed]:
 # Compute the first 15 Fibonacci numbers and their sum

def fibonacci(n):
    fib_seq = [0, 1]
    for i in range(2, n):
        fib_seq.append(fib_seq[-1] + fib_seq[-2])
    return fib_seq

# First 15 Fibonacci numbers
fib_15 = fibonacci(15)

# Their sum
fib_sum = sum(fib_15)

fib_15, fib_sum 

[AI]: The first 15 Fibonacci numbers are:
0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377

Their sum is **986**.


## ۷. مقایسه‌ی روش‌ها و جمع‌بندی

| روش | اجرا کجاست؟ | امنیت | مناسب برای |
|---|---|---|---|
| `PythonREPLTool` | سیستم شما | ⚠️ ضعیف | آموزش، تست سریع |
| `PythonAstREPLTool` | سیستم شما | ⚠️ ضعیف | تحلیل DataFrame تو نوت‌بوک شخصی |
| `run_python` (subprocess + timeout) | سیستم شما | ⚠️ متوسط | جلوگیری از loop بی‌نهایت |
| Code Interpreter OpenAI | سرور OpenAI | ✅ خوب | محصول واقعی، فایل و نمودار |
| Riza / E2B | sandbox ابری | ✅ خوب | محصول واقعی، مستقل از ارائه‌دهنده‌ی مدل |

**قاعده‌های مهم:**
- به agent بگید حتماً نتیجه رو `print` کنه (به‌جز `PythonAstREPLTool`).
- همیشه timeout بذارید و طول خروجی رو محدود کنید.
- خطا رو به مدل برگردونید تا خودش کدش رو اصلاح کنه.
- کد تولید‌شده توسط LLM رو هرگز روی سرور یا سیستم حاوی اطلاعات حساس بدون sandbox اجرا نکنید.

## منابع
* [لیست tool ها و integration ها](https://docs.langchain.com/oss/python/integrations/tools)
* [Riza Code Interpreter در LangChain](https://docs.langchain.com/oss/python/integrations/tools/riza)
* [E2B Data Analysis در LangChain](https://docs.langchain.com/oss/python/integrations/tools/e2b_data_analysis)